# 01. Full Fine-Tuning

**Topics covered:** Dataset Preparation · Trainer API · TrainingArguments

This notebook starts the **fine-tuning** [series](https://github.com/S33mi/modern-ai-llm-journey/tree/main/03_finetuning). We take a pre-trained model and update *all* of its weights on a downstream task.

We will:
1. Prepare a dataset with the 🤗 Datasets library
2. Tokenize and create a data collator
3. Configure **TrainingArguments**
4. Run training with the high-level **Trainer** API
5. Evaluate and save the fine-tuned model

## 1. Setup & Imports
Run once if needed!
```bash
pip install transformers datasets evaluate accelerate scikit-learn
```

In [1]:
# ! pip install transformers datasets evaluate accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.6 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch: {torch.__version__}")

Using device: cpu
PyTorch: 2.11.0+cpu


## 2. Dataset Preparation

We use a small subset of the **IMDB** movie-review sentiment dataset so the notebook runs quickly.

In real projects you would typically:
- load your own CSV / JSON with `load_dataset("csv", data_files=...)`
- or push a private dataset to the Hub
- or take any suitable dataset

In [3]:
# Load a small slice for demonstration (full IMDB is ~25k train examples) in case you have memory issues
raw = load_dataset("acosio14/imbd-movie-reviews") # acosio14/imbd-movie-reviews, Amanprime/IMBD-DATASET

# Take a tiny subset so training finishes in minutes even on CPU
# train_ds = raw["train"].shuffle(seed=42).select(range(2000))
# eval_ds  = raw["test"].shuffle(seed=42).select(range(500))

# print(train_ds)
# print("\nSample:")
# print(train_ds[0])

# Split the only available split into train + test (to aviode multifile csv error)
split = raw["train"].train_test_split(test_size=0.1, seed=42)   # 10 % for eval

# Take smaller subsets if you want faster runs
train_ds = split["train"].shuffle(seed=42).select(range(2000))
eval_ds  = split["test"].shuffle(seed=42).select(range(500))

print(train_ds)
print("\nSample:")
print(train_ds[0])

README.md:   0%|          | 0.00/491 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 49.4MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 12.2MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/40000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset({
    features: ['text', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 2000
})

Sample:
{'text': 'I am not going to spoil the contents to anyone, who has not yet watched this humble masterpiece by Kay Pollak.<br /><br />A world famous conductor brilliantly played by Michael Nyqvist seeks peace from stress by moving back to his childhood village. The villagers, who has followed the genius in silence, are slowly tempting him to share of his greatness.<br /><br />Each role in this movie, has a very specific purpose and shows a remarkable potential in each of the actors playing their own chord in short but precise words, a symphony of love.<br /><br />Not love in the sense of relationship, but in the tone of the spirit deeply buried within each of the characters, each revealing their own present story, their needs, their skeletons, desires and much more.<br /><br />I shall not forget to mention, the two main parts played by Frida Hallgren and Michael Nyqvist, whose dramas

### Tokenization

We map the tokenizer over the dataset. Setting `batched=True` makes it much faster.

In [4]:
model_name = "distilbert-base-uncased"  # small & fast
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

train_tok = train_ds.map(tokenize, batched=True, remove_columns=["text"])
eval_tok  = eval_ds.map(tokenize, batched=True, remove_columns=["text"])

# Rename label column if needed (IMDB already uses "label")
print(train_tok)
print("Columns:", train_tok.column_names)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'attention_mask', 'token_type_ids'],
    num_rows: 2000
})
Columns: ['labels', 'input_ids', 'attention_mask', 'token_type_ids']


### Data Collator

`DataCollatorWithPadding` dynamically pads each batch to the longest sequence in that batch (more efficient than padding everything to `max_length`).

In [5]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 3. Load a Pre-trained Model for Classification

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,                     # positive / negative
)
model = model.to(device)
print(model.config)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_dim": 3072,
  "initializer_range": 0.02,
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.16.1",
  "vocab_size": 30522
}



## 4. Metrics

In [7]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

## 5. TrainingArguments

`TrainingArguments` controls virtually every aspect of the training loop.

Important fields:

| Argument | Meaning |
|----------|--------|
| `output_dir` | Where checkpoints & logs are written |
| `num_train_epochs` | Full passes over the training set |
| `per_device_train_batch_size` | Batch size per GPU / CPU |
| `per_device_eval_batch_size` | Eval batch size |
| `learning_rate` | Peak learning rate |
| `weight_decay` | L2 regularisation |
| `evaluation_strategy`/`eval_strategy` | `"no"` / `"steps"` / `"epoch"` |
| `save_strategy` | When to write checkpoints |
| `load_best_model_at_end` | Restore the best checkpoint after training |
| `logging_steps` | How often to log |
| `fp16` / `bf16` | Mixed-precision training |

In [8]:
training_args = TrainingArguments(
    output_dir="./results-imdb-full",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",          # ← changed from evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=50,
    fp16=torch.cuda.is_available(),   # use mixed precision on GPU
    report_to="none",                 # disable wandb / tensorboard for simplicity
)

print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start

## 6. The Trainer API

`Trainer` wires together the model, data, arguments, collator and metrics into a complete training loop (including gradient accumulation, mixed precision, distributed training, etc.).

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    processing_class=tokenizer,   # ← changed from tokenizer=...
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [10]:
# Run training (a few minutes on GPU, longer on CPU)
train_result = trainer.train()
print(train_result)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.372079,0.297338,0.868000,0.872587
2,0.225370,0.405178,0.866000,0.870906


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.3312343063354492, metrics={'train_runtime': 4467.165, 'train_samples_per_second': 0.895, 'train_steps_per_second': 0.112, 'total_flos': 264748515032640.0, 'train_loss': 0.3312343063354492, 'epoch': 2.0})


## 7. Evaluation

In [14]:
metrics = trainer.evaluate()
print("Eval metrics:", metrics)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.225370,0.297338,2,0.868000,0.872587


Eval metrics: {'eval_loss': 0.2973376214504242, 'eval_accuracy': 0.868, 'eval_f1': 0.8725868725868726}


## 8. Inference with the Fine-Tuned Model

In [15]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    pred = logits.argmax(-1).item()
    label = "positive" if pred == 1 else "negative"
    prob = torch.softmax(logits, dim=-1)[0, pred].item()
    return label, prob


examples = [
    "This movie was an absolute masterpiece. I loved every minute of it!",
    "Boring, predictable and a complete waste of time.",
    "The acting was decent but the plot made no sense.",
]

for text in examples:
    label, prob = predict(text)
    print(f"[{label} {prob:.2f}]  {text}")

[positive 0.95]  This movie was an absolute masterpiece. I loved every minute of it!
[negative 0.95]  Boring, predictable and a complete waste of time.
[negative 0.94]  The acting was decent but the plot made no sense.


## 9. Saving & Reloading

In [16]:
save_dir = "./my-imdb-model"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"Model saved to {save_dir}")

# Reload later with:
# model = AutoModelForSequenceClassification.from_pretrained(save_dir)
# tokenizer = AutoTokenizer.from_pretrained(save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./my-imdb-model


### Download the model for further use if needed

Run this code for optional download

```
import shutil
from google.colab import files

save_dir = "./my-imdb-model"

# Create a zip of the model folder
zip_path = shutil.make_archive("my-imdb-model", "zip", save_dir)

print(f"Created: {zip_path}")

# Download the zip file to your browser
files.download(zip_path)
```

## 10. Full Fine-Tuning – Pros & Cons

| | Full Fine-Tuning |
|---|------------------|
| **Pros** | Highest possible quality; simple mental model |
| **Cons** | Expensive (memory & compute); risk of catastrophic forgetting; one checkpoint per task |
| **When to use** | Small models, abundant GPU memory, single-task deployment |

For large models (billions of parameters) we almost always prefer **parameter-efficient** methods such as LoRA / QLoRA – covered in the next notebooks.

## 11. Summary

| Step | Tool |
|------|------|
| Load data | `datasets.load_dataset` |
| Tokenize | `dataset.map(tokenize, batched=True)` |
| Dynamic padding | `DataCollatorWithPadding` |
| Hyper-parameters | `TrainingArguments` |
| Training loop | `Trainer` |
| Metrics | `evaluate` library + `compute_metrics` |
| Save | `trainer.save_model` / `tokenizer.save_pretrained` |

### Canonical training snippet

```python
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    learning_rate=2e-5,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()
```

---

**Next notebook:** [`02_lora_finetuning.ipynb`](
(https://github.com/S33mi/modern-ai-llm-journey/tree/main/03_finetuning/02_lora_finetuning.ipynb)  
LoRA Theory · Rank · Alpha · PEFT Library

---

**For contribution and insihght:** [**S33mi**](https://github.com/S33mi)

Open to Data Analytics and ML/AL related opportunities